# run_final_xai_results.ipynb

This notebook generates the result files needed by `cka_sat_gaas_analysis.ipynb`.

It saves:

- `saved_outputs/cka_results.csv`
- `saved_outputs/sat_results.csv`
- `saved_outputs/gaas_results.csv`
- `saved_outputs/transferability_matrix.csv`
- `saved_outputs/transferability_long.csv`
- `saved_outputs/asr_results.csv`
- `saved_outputs/model_info.csv`
- `saved_outputs/selected_layers.csv`
- `saved_outputs/transferable_example_cka.csv`

Run this notebook first. Then run the analysis notebook.

## 1. Imports and settings

In [1]:
from pathlib import Path
from collections import OrderedDict
import itertools
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x

from robustbench.utils import load_model
from robustbench.data import load_cifar10
from robustbench.model_zoo.enums import ThreatModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SAVED_DIR = Path("saved_outputs")
SAVED_DIR.mkdir(parents=True, exist_ok=True)

DATASET = "cifar10"
EPS = 8 / 255

# Start small. Increase after the notebook works.
N_EXAMPLES = 64
BATCH_SIZE = 16

# PGD is faster than full AutoAttack and is enough to generate first project results.
PGD_STEPS = 10
PGD_ALPHA = 2 / 255

CKA_N_EXAMPLES = 64
GAAS_N_EXAMPLES = 32

print("Device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Saved outputs:", SAVED_DIR.resolve())

KeyboardInterrupt: 

## 2. Models and metadata

Edit this list if one model fails to load.

In [ ]:
MODEL_CONFIGS = [
    {
        "model": "Standard",
        "architecture": "ResNet",
        "training_type": "standard",
    },
    {
        "model": "Engstrom2019Robustness",
        "architecture": "ResNet",
        "training_type": "adversarial_training",
    },
    {
        "model": "Wong2020Fast",
        "architecture": "WideResNet",
        "training_type": "fast_adversarial_training",
    },
    {
        "model": "Rice2020Overfitting",
        "architecture": "WideResNet",
        "training_type": "adversarial_training",
    },
]

model_info_df = pd.DataFrame(MODEL_CONFIGS)
model_info_path = SAVED_DIR / "model_info.csv"
model_info_df.to_csv(model_info_path, index=False)

display(model_info_df)
print("Saved:", model_info_path)

## 3. Load CIFAR-10 subset

In [ ]:
x_test, y_test = load_cifar10(n_examples=N_EXAMPLES, data_dir="./data")
x_test = x_test.to(DEVICE)
y_test = y_test.to(DEVICE)

print("x_test:", x_test.shape, x_test.dtype, x_test.device)
print("y_test:", y_test.shape, y_test.dtype, y_test.device)

## 4. Load models

In [ ]:
def safe_model_name(name):
    return name.replace("/", "_").replace(" ", "_").replace(".", "_")


def load_all_models(model_configs):
    models = OrderedDict()

    for cfg in model_configs:
        model_name = cfg["model"]
        print("=" * 80)
        print("Loading:", model_name)

        model = load_model(
            model_name=model_name,
            dataset=DATASET,
            threat_model=ThreatModel.Linf,
        ).to(DEVICE)

        model.eval()
        models[model_name] = model
        print("Loaded:", model_name)

    return models


models = load_all_models(MODEL_CONFIGS)

print("Loaded models:")
for name in models:
    print("-", name)

## 5. Prediction and PGD helper functions

In [ ]:
def batched_predict(model, x, batch_size=BATCH_SIZE):
    preds = []
    model.eval()

    with torch.no_grad():
        for start in range(0, x.shape[0], batch_size):
            xb = x[start:start + batch_size]
            logits = model(xb)
            preds.append(logits.argmax(dim=1).detach().cpu())

    return torch.cat(preds, dim=0)


def batched_accuracy(model, x, y, batch_size=BATCH_SIZE):
    preds = batched_predict(model, x, batch_size=batch_size).to(y.device)
    return (preds == y).float().mean().item()


def pgd_linf_attack(model, images, labels, eps=EPS, alpha=PGD_ALPHA, steps=PGD_STEPS):
    model.eval()

    images = images.detach()
    labels = labels.detach()
    original = images.detach()

    adv = original + torch.empty_like(original).uniform_(-eps, eps)
    adv = torch.clamp(adv, 0, 1)

    for _ in range(steps):
        adv.requires_grad_(True)
        logits = model(adv)
        loss = F.cross_entropy(logits, labels)
        grad = torch.autograd.grad(loss, adv)[0]

        adv = adv.detach() + alpha * torch.sign(grad.detach())
        delta = torch.clamp(adv - original, min=-eps, max=eps)
        adv = torch.clamp(original + delta, 0, 1).detach()

    return adv


def get_or_create_pgd_adversarial(model_name, model, x, y):
    adv_path = SAVED_DIR / f"x_adv_pgd_{safe_model_name(model_name)}_eps8_255_n{len(x)}.pt"

    if adv_path.exists():
        print(f"Loading cached PGD adversarial examples: {adv_path}")
        saved = torch.load(adv_path, map_location=DEVICE)
        return saved["x_adv"].to(DEVICE)

    print(f"Creating PGD adversarial examples for {model_name}")
    adv_batches = []

    for start in tqdm(range(0, x.shape[0], BATCH_SIZE), desc=f"PGD {model_name}"):
        xb = x[start:start + BATCH_SIZE]
        yb = y[start:start + BATCH_SIZE]
        adv_b = pgd_linf_attack(model, xb, yb)
        adv_batches.append(adv_b.detach().cpu())

    x_adv = torch.cat(adv_batches, dim=0).to(DEVICE)

    torch.save(
        {
            "model_name": model_name,
            "eps": EPS,
            "pgd_steps": PGD_STEPS,
            "pgd_alpha": PGD_ALPHA,
            "x_adv": x_adv.detach().cpu(),
            "y": y.detach().cpu(),
        },
        adv_path,
    )

    print("Saved:", adv_path)
    return x_adv

## 6. Generate ASR results

Creates `saved_outputs/asr_results.csv`.

In [ ]:
asr_rows = []
adv_cache = {}

for model_name, model in models.items():
    print("=" * 80)
    print("Evaluating:", model_name)

    clean_acc = batched_accuracy(model, x_test, y_test)

    x_adv = get_or_create_pgd_adversarial(model_name, model, x_test, y_test)
    adv_cache[model_name] = x_adv

    robust_acc = batched_accuracy(model, x_adv, y_test)
    asr = 1.0 - robust_acc

    row = {
        "model": model_name,
        "clean_accuracy": clean_acc,
        "robust_accuracy_pgd": robust_acc,
        "asr_pgd": asr,
        "n_examples": N_EXAMPLES,
        "eps": EPS,
        "pgd_steps": PGD_STEPS,
    }

    asr_rows.append(row)
    print(row)

asr_df = pd.DataFrame(asr_rows)
asr_path = SAVED_DIR / "asr_results.csv"
asr_df.to_csv(asr_path, index=False)

display(asr_df)
print("Saved:", asr_path)

## 7. Transferability and SAT results

Creates:

- `saved_outputs/transferability_long.csv`
- `saved_outputs/transferability_matrix.csv`
- `saved_outputs/sat_results.csv`

Here, `sat` is saved as the empirical transfer success rate. You can replace it later if your group uses a stricter SAT formula.

In [ ]:
transfer_rows = []

clean_preds = {
    name: batched_predict(model, x_test).to(DEVICE)
    for name, model in models.items()
}

for source_name, x_adv_source in adv_cache.items():
    source_clean_pred = clean_preds[source_name]
    source_adv_pred = batched_predict(models[source_name], x_adv_source).to(DEVICE)

    source_success_mask = (source_clean_pred == y_test) & (source_adv_pred != y_test)
    source_success_total = int(source_success_mask.sum().item())

    print(f"{source_name}: source-success examples = {source_success_total}/{N_EXAMPLES}")

    for target_name, target_model in models.items():
        target_adv_pred = batched_predict(target_model, x_adv_source).to(DEVICE)
        target_fooled_mask = target_adv_pred != y_test

        if source_success_total == 0:
            transfer_asr = np.nan
            transfer_success_count = 0
        else:
            transfer_success_count = int((source_success_mask & target_fooled_mask).sum().item())
            transfer_asr = transfer_success_count / source_success_total

        transfer_rows.append({
            "source_model": source_name,
            "target_model": target_name,
            "source_success_count": source_success_total,
            "transfer_success_count": transfer_success_count,
            "transfer_asr": transfer_asr,
            "sat": transfer_asr,
            "n_examples": N_EXAMPLES,
            "eps": EPS,
            "pgd_steps": PGD_STEPS,
        })

transfer_long_df = pd.DataFrame(transfer_rows)

transfer_long_path = SAVED_DIR / "transferability_long.csv"
transfer_long_df.to_csv(transfer_long_path, index=False)

transfer_matrix_df = transfer_long_df.pivot(
    index="source_model",
    columns="target_model",
    values="transfer_asr"
).reset_index()

transfer_matrix_path = SAVED_DIR / "transferability_matrix.csv"
transfer_matrix_df.to_csv(transfer_matrix_path, index=False)

sat_df = transfer_long_df[["source_model", "target_model", "sat"]].copy()
sat_path = SAVED_DIR / "sat_results.csv"
sat_df.to_csv(sat_path, index=False)

display(transfer_long_df)
display(transfer_matrix_df)

print("Saved:", transfer_long_path)
print("Saved:", transfer_matrix_path)
print("Saved:", sat_path)

## 8. Automatic layer selection for CKA

This selects representative Conv2d/Linear layers from every model.

Important: `source_layer` and `target_layer` are saved as numbers, so same-depth layer-wise CKA can be analyzed even if real module names differ.

In [ ]:
def auto_select_layers(model, max_layers=6):
    candidates = []

    for name, module in model.named_modules():
        if name == "":
            continue

        if isinstance(module, (nn.Conv2d, nn.Linear)):
            candidates.append((name, module))

    if len(candidates) == 0:
        raise ValueError("No Conv2d/Linear layers found.")

    if len(candidates) <= max_layers:
        selected = candidates
    else:
        indices = np.linspace(0, len(candidates) - 1, max_layers).round().astype(int)
        selected = [candidates[i] for i in indices]

    return selected


selected_layers = {}

for model_name, model in models.items():
    selected = auto_select_layers(model, max_layers=6)

    selected_layers[model_name] = [
        {
            "layer_index": i + 1,
            "layer_name": name,
            "module_type": type(module).__name__,
        }
        for i, (name, module) in enumerate(selected)
    ]

selected_layers_df = pd.DataFrame([
    {"model": model_name, **layer_info}
    for model_name, layers in selected_layers.items()
    for layer_info in layers
])

selected_layers_path = SAVED_DIR / "selected_layers.csv"
selected_layers_df.to_csv(selected_layers_path, index=False)

display(selected_layers_df)
print("Saved:", selected_layers_path)

## 9. Feature extraction and CKA functions

In [ ]:
class FeatureExtractor:
    def __init__(self, model, layer_infos):
        self.model = model
        self.layer_infos = layer_infos
        self.features = OrderedDict()
        self.handles = []

        modules = dict(model.named_modules())

        for info in layer_infos:
            layer_index = info["layer_index"]
            layer_name = info["layer_name"]

            if layer_name not in modules:
                raise ValueError(f"Layer not found: {layer_name}")

            module = modules[layer_name]
            handle = module.register_forward_hook(self._make_hook(layer_index))
            self.handles.append(handle)

    def _make_hook(self, layer_index):
        def hook(module, inputs, output):
            if isinstance(output, tuple):
                output = output[0]
            self.features[layer_index] = output.detach()
        return hook

    def clear(self):
        self.features = OrderedDict()

    def remove(self):
        for handle in self.handles:
            handle.remove()


def flatten_features(x):
    if x.ndim > 2:
        x = torch.flatten(x, start_dim=1)
    return x


def collect_features(model_name, model, x):
    layer_infos = selected_layers[model_name]
    extractor = FeatureExtractor(model, layer_infos)

    collected = {info["layer_index"]: [] for info in layer_infos}

    model.eval()
    with torch.no_grad():
        for start in range(0, x.shape[0], BATCH_SIZE):
            xb = x[start:start + BATCH_SIZE]
            extractor.clear()
            _ = model(xb)

            for layer_index in collected:
                feat = flatten_features(extractor.features[layer_index]).detach().cpu()
                collected[layer_index].append(feat)

    extractor.remove()

    return {
        layer_index: torch.cat(chunks, dim=0)
        for layer_index, chunks in collected.items()
    }


def linear_cka(X, Y):
    X = X.float()
    Y = Y.float()

    n = min(X.shape[0], Y.shape[0])
    X = X[:n]
    Y = Y[:n]

    X = X - X.mean(dim=0, keepdim=True)
    Y = Y - Y.mean(dim=0, keepdim=True)

    hsic = torch.norm(X.T @ Y, p="fro") ** 2
    norm_x = torch.norm(X.T @ X, p="fro")
    norm_y = torch.norm(Y.T @ Y, p="fro")

    denom = norm_x * norm_y

    if denom.item() == 0:
        return np.nan

    return float((hsic / denom).item())

## 10. Compute CKA

Creates `saved_outputs/cka_results.csv`.

In [ ]:
print("Collecting clean features for CKA...")

x_cka = x_test[:CKA_N_EXAMPLES]
feature_cache = {}

for model_name, model in models.items():
    print("Collecting:", model_name)
    feature_cache[model_name] = collect_features(model_name, model, x_cka)

cka_rows = []

for source_name, target_name in itertools.permutations(models.keys(), 2):
    print(f"CKA: {source_name} -> {target_name}")

    source_features = feature_cache[source_name]
    target_features = feature_cache[target_name]

    source_layer_lookup = {
        info["layer_index"]: info["layer_name"]
        for info in selected_layers[source_name]
    }

    target_layer_lookup = {
        info["layer_index"]: info["layer_name"]
        for info in selected_layers[target_name]
    }

    for source_layer_idx, X in source_features.items():
        for target_layer_idx, Y in target_features.items():
            score = linear_cka(X, Y)

            cka_rows.append({
                "source_model": source_name,
                "target_model": target_name,
                "source_layer": source_layer_idx,
                "target_layer": target_layer_idx,
                "source_layer_name": source_layer_lookup[source_layer_idx],
                "target_layer_name": target_layer_lookup[target_layer_idx],
                "cka": score,
                "n_examples": int(min(X.shape[0], Y.shape[0])),
            })

cka_df = pd.DataFrame(cka_rows)
cka_path = SAVED_DIR / "cka_results.csv"
cka_df.to_csv(cka_path, index=False)

display(cka_df.head())
print("Saved:", cka_path)

## 11. Compute GAAS

Creates `saved_outputs/gaas_results.csv`.

This implementation uses gradient cosine similarity as a GAAS-style proxy.

In [ ]:
def input_gradient(model, images, labels):
    model.eval()

    images = images.detach().clone().to(DEVICE)
    labels = labels.detach().clone().to(DEVICE)

    images.requires_grad_(True)
    logits = model(images)
    loss = F.cross_entropy(logits, labels)

    grad = torch.autograd.grad(loss, images)[0]
    return grad.detach()


def gradient_cosine_similarity(grad_a, grad_b):
    ga = grad_a.flatten(start_dim=1)
    gb = grad_b.flatten(start_dim=1)

    sim = F.cosine_similarity(ga, gb, dim=1)
    return sim.detach().cpu().numpy()


x_gaas = x_test[:GAAS_N_EXAMPLES]
y_gaas = y_test[:GAAS_N_EXAMPLES]

gradient_cache = {}

for model_name, model in models.items():
    print("Computing gradients:", model_name)
    grads = []

    for start in range(0, x_gaas.shape[0], BATCH_SIZE):
        xb = x_gaas[start:start + BATCH_SIZE]
        yb = y_gaas[start:start + BATCH_SIZE]
        grad_b = input_gradient(model, xb, yb)
        grads.append(grad_b.detach().cpu())

    gradient_cache[model_name] = torch.cat(grads, dim=0).to(DEVICE)

gaas_rows = []

for source_name, target_name in itertools.permutations(models.keys(), 2):
    sims = gradient_cosine_similarity(
        gradient_cache[source_name],
        gradient_cache[target_name]
    )

    gaas_rows.append({
        "source_model": source_name,
        "target_model": target_name,
        "gaas": float(np.mean(sims)),
        "gaas_std": float(np.std(sims)),
        "n_examples": int(len(sims)),
    })

gaas_df = pd.DataFrame(gaas_rows)
gaas_path = SAVED_DIR / "gaas_results.csv"
gaas_df.to_csv(gaas_path, index=False)

display(gaas_df)
print("Saved:", gaas_path)

## 12. Transferable vs non-transferable CKA

Creates `saved_outputs/transferable_example_cka.csv`.

This is for the advanced divergence/reconvergence research question.

Note: CKA is calculated on groups of transferable/non-transferable adversarial examples.

In [ ]:
transferable_cka_rows = []
MIN_GROUP_SIZE = 2

for source_name, target_name in itertools.permutations(models.keys(), 2):
    print(f"Transferable/non-transferable CKA: {source_name} -> {target_name}")

    source_model = models[source_name]
    target_model = models[target_name]
    x_adv_source = adv_cache[source_name]

    source_clean_pred = clean_preds[source_name]
    source_adv_pred = batched_predict(source_model, x_adv_source).to(DEVICE)
    target_adv_pred = batched_predict(target_model, x_adv_source).to(DEVICE)

    source_success_mask = (source_clean_pred == y_test) & (source_adv_pred != y_test)
    target_fooled_mask = target_adv_pred != y_test

    transferable_mask = source_success_mask & target_fooled_mask
    non_transferable_mask = source_success_mask & (~target_fooled_mask)

    groups = {
        1: transferable_mask,
        0: non_transferable_mask,
    }

    for transferable_value, mask in groups.items():
        idx = torch.where(mask)[0]

        if len(idx) < MIN_GROUP_SIZE:
            print(f"  Skipping transferable={transferable_value}, only {len(idx)} examples")
            continue

        x_group = x_adv_source[idx]

        source_features = collect_features(source_name, source_model, x_group)
        target_features = collect_features(target_name, target_model, x_group)

        source_layer_lookup = {
            info["layer_index"]: info["layer_name"]
            for info in selected_layers[source_name]
        }

        target_layer_lookup = {
            info["layer_index"]: info["layer_name"]
            for info in selected_layers[target_name]
        }

        for source_layer_idx, X in source_features.items():
            for target_layer_idx, Y in target_features.items():
                score = linear_cka(X, Y)

                transferable_cka_rows.append({
                    "source_model": source_name,
                    "target_model": target_name,
                    "sample_id": -1,
                    "source_layer": source_layer_idx,
                    "target_layer": target_layer_idx,
                    "source_layer_name": source_layer_lookup[source_layer_idx],
                    "target_layer_name": target_layer_lookup[target_layer_idx],
                    "cka": score,
                    "transferable": int(transferable_value),
                    "n_samples": int(len(idx)),
                })

transferable_example_cka_df = pd.DataFrame(transferable_cka_rows)
transferable_example_cka_path = SAVED_DIR / "transferable_example_cka.csv"
transferable_example_cka_df.to_csv(transferable_example_cka_path, index=False)

display(transferable_example_cka_df.head())
print("Saved:", transferable_example_cka_path)

## 13. Verify outputs

In [ ]:
required_files = [
    "cka_results.csv",
    "sat_results.csv",
    "gaas_results.csv",
    "transferability_matrix.csv",
    "transferability_long.csv",
    "asr_results.csv",
    "model_info.csv",
    "selected_layers.csv",
    "transferable_example_cka.csv",
]

for filename in required_files:
    path = SAVED_DIR / filename
    print(f"{filename:35s} exists = {path.exists()}")

## 14. Next step

After this notebook finishes, run:

```text
cka_sat_gaas_analysis.ipynb
```

That notebook loads the CSV files and creates the final plots/tables for your report.